FMCG Research Copilot — P3 RAG: Colab Test Notebook

How to use:
  Step 1. Upload p3-retrieval folder to Google Drive at:
          MyDrive/fmcg-research-copilot/p3-retrieval/
          (Must include: index/, parsed/, retrieve.py, requirements.txt)
  Step 2. Upload THIS .ipynb file to Colab (File → Upload notebook)
  Step 3. Run all cells top to bottom.

### Cell 1 — Install dependencies (run once per session, ~2 min)

In [ ]:
import subprocess, sys

packages = [
    "faiss-cpu",
    "sentence-transformers",
    "rank-bm25",
    "python-dotenv",
    "httpx",
    "tqdm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("Packages installed.")

### Cell 2 — IITK API credentials (optional but faster)

In [ ]:
import os

# If you have the IITK embedding key, paste it here.
# retrieve() will use the API (~1s) instead of local model (~15s cold start).
# Both use bge-small-en-v1.5 so results are identical.
# Leave IITK_EMBED_KEY empty to fall back to local SentenceTransformer.

os.environ["IITK_EMBED_URL"] = "https://103.42.50.41:1035/v1/embeddings"
os.environ["IITK_EMBED_KEY"] = ""   # ← paste key from shared-embedding-key.txt
os.environ["CACERT_PATH"]    = ""   # ← full path to vm02-ca.crt, or leave empty

### Cell 3 — Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

# Adjust if you placed the folder elsewhere in your Drive
P3_DIR = Path("/content/drive/MyDrive/fmcg-research-copilot/p3-retrieval")

if not P3_DIR.exists():
    raise FileNotFoundError(f"p3-retrieval not found at {P3_DIR}. Check Drive path above.")

sys.path.insert(0, str(P3_DIR))
print(f"Using P3 at: {P3_DIR}")
print(f"  index dir:  {P3_DIR / 'index'} — exists: {(P3_DIR / 'index').exists()}")
print(f"  parsed dir: {P3_DIR / 'parsed'} — exists: {(P3_DIR / 'parsed').exists()}")

### Cell 4 — Load index and models (~15–30s first time)

In [ ]:
import retrieve as _r
from pathlib import Path

# Patch paths so retrieve.py finds index/ and parsed/ in Drive
P3_DIR = Path("/content/drive/MyDrive/fmcg-research-copilot/p3-retrieval")
_r.RAG_DIR    = P3_DIR
_r.INDEX_DIR  = P3_DIR / "index"
_r.PARSED_DIR = P3_DIR / "parsed"

from retrieve import retrieve, get_table, _load_models, _load_index

print("Loading index and models…")
_load_models()
_load_index()

from retrieve import _chunk_map
print(f"Ready: {len(_chunk_map):,} chunks loaded.")

### Cell 5 — search() and show_table() helpers

In [ ]:
def search(
    query: str,
    company: str = None,
    fiscal_year: str = None,
    doc_type: str = None,
    section: str = None,
    k: int = 5,
    show_text_chars: int = 400,
):
    """
    Args:
        query:        Natural language question.
        company:      "HUL", "Dabur", "ITC", "Nestle India", "Britannia",
                      "Marico", "GCPL", "Colgate-Palmolive India",
                      "Tata Consumer Products", "Emami"
        fiscal_year:  "FY24", "FY23", "FY22" … "FY17"
        doc_type:     "annual_report", "quarterly_results", "investor_presentation"
        section:      "P&L", "Balance Sheet", "MD&A", "Notes to Accounts",
                      "Segment Results", "Cash Flow"
        k:            Number of results (default 5)
    """
    filters = {}
    if company:      filters["company"]     = company
    if fiscal_year:  filters["fiscal_year"] = fiscal_year
    if doc_type:     filters["doc_type"]    = doc_type
    if section:      filters["section"]     = section

    print(f"\n{'='*70}")
    print(f"QUERY: {query!r}")
    if filters:
        print(f"FILTERS: {filters}")
    print(f"{'='*70}")

    results = retrieve(query, k=k, filters=filters or None)

    if not results:
        print("No results returned.")
        return results

    for i, r in enumerate(results, 1):
        print(f"\n[{i}] score={r['score']:.3f}  |  {r['company']} · {r['fiscal_year']} · {r['doc_type']}")
        print(f"     chunk: {r['chunk_id']}")
        print(f"     page {r['page']}  section: {r['section']}  basis: {r.get('standalone_consolidated', '?')}")
        raw = r["text"]
        if "<<DATA>>" in raw:
            raw = raw.split("<<DATA>>", 1)[1]
            if "<<END DATA>>" in raw:
                raw = raw.rsplit("<<END DATA>>", 1)[0]
            lines = raw.strip().splitlines()
            if lines and lines[0].startswith("[Source:"):
                raw = "\n".join(lines[1:]).strip()
        print(f"     ···\n     {raw[:show_text_chars].replace(chr(10), chr(10)+'     ')}")
        if len(raw) > show_text_chars:
            print(f"     … (+{len(raw)-show_text_chars} more chars)")

    return results


def show_table(doc_id: str, page: int = None, section_hint: str = None):
    """Pass doc_id from a search() result to extract a table."""
    result = get_table(doc_id, page=page, section_hint=section_hint)
    print(f"\nTABLE from {doc_id}  page={result.get('page')}  section={result.get('section')}")
    if result.get("low_confidence"):
        print("(low confidence — showing raw text fallback)")
        print(result.get("text_fallback", "(no fallback)"))
        return result
    headers = result.get("headers", [])
    rows    = result.get("rows", [])
    if headers:
        col_w = [max(len(h), max((len(str(r[i])) for r in rows if i < len(r)), default=0))
                 for i, h in enumerate(headers)]
        fmt = "  ".join(f"{{:<{w}}}" for w in col_w)
        print("  " + fmt.format(*headers))
        print("  " + "  ".join("-" * w for w in col_w))
        for row in rows:
            padded = list(row) + [""] * (len(headers) - len(row))
            print("  " + fmt.format(*padded))
    return result

print("search() and show_table() ready.")

### Cell 6 — Sample queries (edit and re-run freely)

In [ ]:
# Single-company metric lookup
r1 = search(
    "HUL advertising and promotion expenses FY24",
    company="HUL", fiscal_year="FY24",
)

# Cross-company comparison
r2 = search(
    "revenue from operations across FMCG companies FY24",
    fiscal_year="FY24", doc_type="annual_report",
)

# Management commentary
r3 = search(
    "Dabur rural distribution strategy and growth outlook",
    company="Dabur",
)

# Try get_table on the top result from r1
if r1:
    show_table(r1[0]["doc_id"], page=r1[0]["page"], section_hint="P&L")

### Cell 7 — Your own query (edit and re-run)

In [ ]:
MY_QUERY   = "Nestle India revenue and profit growth FY24"
MY_COMPANY = "Nestle India"   # set to None to search all companies
MY_FY      = "FY24"           # set to None to search all years

results = search(MY_QUERY, company=MY_COMPANY, fiscal_year=MY_FY, k=5)